# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



### Run SQl using DuckDB

In [1]:
# Download the Chinook sqlite database
%%bash
[ -f chinook.zip ] || curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -o chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [2]:
#Remove any old copy, then unzip
!rm -f chinook.db
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [3]:
# Check the files
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 15 14:23 chinook.zip
drwxr-xr-x 1 root root   4096 Nov 12 16:40 sample_data


In [4]:
# Import Python libraries

import duckdb
import pandas as pd
import sqlite3 as db
import contextlib



con = duckdb.connect()                                  # an empty DuckDB session in memory
con.sql("INSTALL sqlite; LOAD sqlite;")                 # add-on that lets DuckDB read SQLite files
con.sql("ATTACH 'chinook.db' AS chinook (TYPE sqlite)") # plug in your file and call it 'chinook'
con.sql("USE chinook")                                  # make it the default, so you can write 'albums'

In [5]:
# check: should list 11 tables + 2 sqlite_ tables
con.sql("SHOW TABLES").df()

,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoice_items
6,invoices
7,media_types
8,playlist_track
9,playlists


## How many columns

In [6]:
#====================================================
# HOW MANY COLUMN IN EACH TABLE ?
#(note show what change if have diffrent database)
#====================================================

con.sql("""
    select table_name,                          -- never changes
           count(*) as n_columns                -- never changes (the name n_columns is your choice)
    from information_schema.columns             -- never changes
    where table_catalog = 'chinook'             -- CHANGE: your ATTACH nickname
      and table_name not like 'sqlite_%'        -- keep only for SQLite files
    group by table_name                         -- never changes
    order by table_name                         -- never changes
""").df()

,table_name,n_columns
0,albums,3
1,artists,2
2,customers,13
3,employees,15
4,genres,2
5,invoice_items,5
6,invoices,9
7,media_types,2
8,playlist_track,2
9,playlists,2


## How many rows

In [7]:
# ============================================================
# HOW MANY ROWS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Idea: ask "how many rows?" once per table, then stack all
#       the answers into one result table.
#
# TO REUSE ON ANOTHER DATABASE:
#   1. Run con.sql("SHOW TABLES").df() to get the table names.
#   2. Make one select line per table.
#   3. On each line, change the name in TWO places:
#        'label'      -> text shown in the output (in quotes)
#        from table   -> the real table being counted (no quotes)
#   4. Put "union all" BETWEEN lines (not before the first one,
#      not after the last one).
# ============================================================

con.sql("""
    -- Each select line answers: "How many rows are in this table?"
    --   'albums'           = a text label we type, so we know which table the number belongs to
    --   as table_name      = names the label column
    --   count(*)           = counts every row, even rows with blank values
    --   as n_rows          = names the count column
    --   from albums        = the table being counted (must match the label)

    select 'albums'         as table_name, count(*) as n_rows from albums
    union all                                   -- "stack the next answer under this one"
    select 'artists'        as table_name, count(*) as n_rows from artists
    union all
    select 'customers'      as table_name, count(*) as n_rows from customers
    union all
    select 'employees'      as table_name, count(*) as n_rows from employees
    union all
    select 'genres'         as table_name, count(*) as n_rows from genres
    union all
    select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items
    union all
    select 'invoices'       as table_name, count(*) as n_rows from invoices
    union all
    select 'media_types'    as table_name, count(*) as n_rows from media_types
    union all
    select 'playlist_track' as table_name, count(*) as n_rows from playlist_track
    union all
    select 'playlists'      as table_name, count(*) as n_rows from playlists
    union all
    select 'tracks'         as table_name, count(*) as n_rows from tracks

    order by table_name                         -- sort the final stacked table A to Z (written once, at the end)
""").df()                                       # .df() only DISPLAYS the result as a pandas table; SQL did the counting

# EASY MISTAKE: changing the 'label' but not the "from" table.
#   You'd get the wrong number under the right name, and NO error.

,table_name,n_rows
0,albums,347
1,artists,275
2,customers,59
3,employees,8
4,genres,25
5,invoice_items,2240
6,invoices,412
7,media_types,5
8,playlist_track,8715
9,playlists,18


## Breaking down the columns

### Identifier Columns

##### Find primary keys

In [8]:
# ============================================================
# WHICH COLUMNS ARE PRIMARY KEYS?
# ------------------------------------------------------------
# duckdb_constraints() is DuckDB's built-in list of the "rules"
# on each table: primary keys, NOT NULL, and so on.
# We keep only the PRIMARY KEY rules.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           unnest(constraint_column_names) as column_name   -- unnest = split a list into one row per item
    from duckdb_constraints()                               -- built-in list of table rules
    where database_name = 'chinook'                         -- only our database
      and constraint_type = 'PRIMARY KEY'                   -- only primary-key rules
    order by table_name
""").df()

,table_name,column_name
0,albums,AlbumId
1,artists,ArtistId
2,customers,CustomerId
3,employees,EmployeeId
4,genres,GenreId
5,invoice_items,InvoiceLineId
6,invoices,InvoiceId
7,media_types,MediaTypeId
8,playlist_track,PlaylistId
9,playlist_track,TrackId


##### Find Foreign keys

In [9]:
# ============================================================
# WHICH COLUMNS ARE FOREIGN KEYS (point to another table)?
# ------------------------------------------------------------
# DuckDB does NOT bring SQLite's foreign keys over when it
# attaches the file. So we use sqlite_query() to send a
# question straight to SQLite and get its answer back.
#
# Everything inside the single quotes '...' is SQLite's SQL.
# Inside that, '' (two single quotes) = one quote mark.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
#           (Only works for SQLite files.)
# ============================================================

con.sql("""
    select table_name, column_name
    from sqlite_query('chinook', '
        select m.name   as table_name,                  -- the table
               f."from" as column_name                  -- the column that points elsewhere
        from sqlite_master as m                          -- SQLite''s list of everything in the file
        join pragma_foreign_key_list(m.name) as f        -- foreign keys for each table
        where m.type = ''table''                         -- tables only (not indexes)
    ')
    order by table_name
""").df()

,table_name,column_name
0,albums,ArtistId
1,customers,SupportRepId
2,employees,ReportsTo
3,invoice_items,TrackId
4,invoice_items,InvoiceId
5,invoices,CustomerId
6,playlist_track,TrackId
7,playlist_track,PlaylistId
8,tracks,MediaTypeId
9,tracks,GenreId


##### Find how many PK,Fk

In [10]:
# ============================================================
# HOW MANY PRIMARY KEYS, FOREIGN KEYS, AND IDENTIFIER COLUMNS
# ARE IN EACH TABLE?
# ------------------------------------------------------------
# "with" builds temporary named tables that only exist while
# this query runs. Here we build three, step by step:
#   1. primary_keys -> every primary-key column, tagged 'PK'
#   2. foreign_keys -> every foreign-key column, tagged 'FK'
#   3. all_keys     -> both lists stacked (UNION ALL keeps every row,
#                      so we can count PKs and FKs separately)
# Then the final select counts each kind per table.
#
# TO REUSE: change 'chinook' in BOTH places to your ATTACH nickname.
# ============================================================

con.sql("""
    with
    -- Step 1: list every primary-key column, and tag it 'PK'
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name,  -- split key lists into one row per column
               'PK' as key_type                                 -- a label we add, so we know what kind of key it is
        from duckdb_constraints()                               -- DuckDB's built-in list of table rules
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 2: list every foreign-key column (asked directly from SQLite), and tag it 'FK'
    foreign_keys as (
        select table_name, column_name, 'FK' as key_type
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 3: stack both lists, keeping every row
    all_keys as (
        select * from primary_keys
        union all
        select * from foreign_keys
    )

    -- Final answer: count each kind of key per table
    select table_name,
           count(*) filter (where key_type = 'PK') as n_pk,                  -- count only the PK rows
           count(*) filter (where key_type = 'FK') as n_fk,                  -- count only the FK rows
           count(distinct column_name)            as n_identifier_columns   -- count each column once, even if it's both
    from all_keys
    group by table_name
    order by table_name
""").df()

,table_name,n_pk,n_fk,n_identifier_columns
0,albums,1,1,2
1,artists,1,0,1
2,customers,1,1,2
3,employees,1,1,2
4,genres,1,0,1
5,invoice_items,1,2,3
6,invoices,1,1,2
7,media_types,1,0,1
8,playlist_track,2,2,2
9,playlists,1,0,1


### Target Columns
**0.** Chinook has no defined prediction target. A target depends on the
question being asked, so none is chosen in this pass.

### Feature Columns

In [11]:
# ============================================================
# HOW MANY FEATURE COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Feature columns = all columns that are NOT identifier columns.
# (No target chosen yet. When you pick one, subtract 1 more.)
#
# "with" builds temporary named tables for this query only:
#   1. all_columns        -> every column in every table
#   2. primary_keys       -> every primary-key column
#   3. foreign_keys       -> every foreign-key column
#   4. identifier_columns -> PK + FK combined, repeats removed
# Then we match all_columns against identifier_columns.
# Whatever does NOT match is a feature column.
#
# TO REUSE: change 'chinook' in all THREE places to your ATTACH nickname.
# ============================================================

con.sql("""
    with
    -- Step 1: every column in every table
    all_columns as (
        select table_name, column_name
        from information_schema.columns                 -- one row per column
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer: match every column against the identifier list
    select a.table_name,
           count(*)                        as n_columns,             -- all columns
           count(i.column_name)            as n_identifier_columns,  -- the ones that matched
           count(*) - count(i.column_name) as n_feature_columns      -- the ones that did NOT match
    from all_columns as a
    left join identifier_columns as i                   -- keep EVERY column, matched or not
           on  a.table_name  = i.table_name             -- same table...
           and a.column_name = i.column_name            -- ...and same column name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_columns,n_identifier_columns,n_feature_columns
0,albums,3,2,1
1,artists,2,1,1
2,customers,13,2,11
3,employees,15,2,13
4,genres,2,1,1
5,invoice_items,5,3,2
6,invoices,9,2,7
7,media_types,2,1,1
8,playlist_track,2,2,0
9,playlists,2,1,1


#### Rows with nulls in each (and categories in each)

In [12]:
# ============================================================
# FOR EACH COLUMN IN ONE TABLE:
#   how many rows are NULL (blank)?
#   how many CATEGORIES (different values) does it have?
# ------------------------------------------------------------
# columns(*) = "do this to EVERY column" (a DuckDB shortcut,
#              so you don't type each column name)
# unpivot    = turn one wide row into one row per column
#              (easier to read: column name | count)
#
# TO REUSE: change the table name (2 places).
# ============================================================

con.sql("""
    with
    -- Step 1: nulls per column -> one wide row, then turned into a list
    null_counts as (
        unpivot (
            select count(*) - count(columns(*))          -- all rows minus non-blank rows = blank rows
            from tracks
        )
        on columns(*)                                    -- turn every column into its own row
        into name column_name value n_null_rows          -- name the two new columns
    ),

    -- Step 2: different values per column -> same idea
    category_counts as (
        unpivot (
            select count(distinct columns(*))            -- how many different values (blanks not counted)
            from tracks
        )
        on columns(*)
        into name column_name value n_categories
    )

    -- Final answer: put the two lists side by side
    select n.column_name,
           n.n_null_rows,
           c.n_categories
    from null_counts as n
    join category_counts as c
      on n.column_name = c.column_name                   -- match each column to itself
    order by n.n_null_rows desc, n.column_name           -- columns with blanks first
""").df()

,column_name,n_null_rows,n_categories
0,Composer,978,852
1,AlbumId,0,347
2,Bytes,0,3501
3,GenreId,0,25
4,MediaTypeId,0,5
5,Milliseconds,0,3080
6,Name,0,3257
7,TrackId,0,3503
8,UnitPrice,0,2


In [13]:
# ============================================================
# NULLS AND CATEGORIES FOR EVERY COLUMN IN EVERY TABLE
# ------------------------------------------------------------
# 1. SQL gives us the list of table names.
# 2. Python fills the SAME query pattern in once per table.
# 3. "union all" stacks them into ONE SQL query, which DuckDB runs.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

# 1. Ask SQL for the table names
tables = con.sql("""
    select table_name
    from information_schema.tables
    where table_catalog = 'chinook'              -- CHANGE: your ATTACH nickname
      and table_name not like 'sqlite_%'         -- keep only for SQLite files
    order by table_name
""").df()["table_name"]

# 2. The query pattern for ONE table. {t} is a blank that gets the table name.
pattern = """
    select '{t}' as table_name, n.column_name, n.n_null_rows, c.n_categories
    from (unpivot (select count(*) - count(columns(*)) from {t})
          on columns(*) into name column_name value n_null_rows) as n
    join (unpivot (select count(distinct columns(*)) from {t})
          on columns(*) into name column_name value n_categories) as c
      on n.column_name = c.column_name
"""

# 3. Fill in the blank for every table, stack with "union all", run once
parts = [pattern.format(t=t) for t in tables]
query = ("select * from (" + "union all".join(parts) + ")"
         + " where n_null_rows > 0"                                  # keep only columns that have blanks
         + " order by table_name, n_null_rows desc, column_name")
con.sql(query).df()

,table_name,column_name,n_null_rows,n_categories
0,customers,Company,49,10
1,customers,Fax,47,12
2,customers,State,29,25
3,customers,PostalCode,4,55
4,customers,Phone,1,58
5,employees,ReportsTo,1,3
6,invoices,BillingState,202,25
7,invoices,BillingPostalCode,28,55
8,tracks,Composer,978,852


#### Data type

In [14]:
# ============================================================
# WHAT DATA TYPES ARE IN THIS DATABASE?
# ------------------------------------------------------------
# Run this first on ANY new database, so you know which type
# names to treat as "numbers" in the next query.
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select data_type,                               -- the kind of value: number, text, date...
           count(*) as n_columns                    -- how many columns have this type
    from information_schema.columns
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    group by data_type
    order by n_columns desc
""").df()

,data_type,n_columns
0,VARCHAR,34
1,BIGINT,24
2,TIMESTAMP,3
3,DOUBLE,3


#### Numerical Columns

In [16]:
# ============================================================
# HOW MANY NUMERICAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Numerical column = a FEATURE column (not an ID) whose
#                    data type is a number type.
#
# Same steps as the feature-column query, plus ONE new thing:
#   is_number -> a true/false flag we add in Step 1
#                (true if the data type is a number type)
#
# TO REUSE: change 'chinook' in all THREE places.
#           If your database has other number types, add them
#           to the is_number list (check with the data-type query).
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus a true/false flag: is it a number type?
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT',
                             'FLOAT', 'DOUBLE', 'DECIMAL') as is_number   -- true or false
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer
    --   i.column_name is null  = no match in the ID list = a feature column
    --   a.is_number            = the column holds numbers
    select a.table_name,
           count(*) filter (where i.column_name is null)                 as n_feature_columns,
           count(*) filter (where i.column_name is null and a.is_number) as n_numerical_columns,
           string_agg(a.column_name, ', ')                                         -- glue the names into one text line
                    filter (where i.column_name is null and a.is_number) as numerical_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_feature_columns,n_numerical_columns,numerical_columns
0,albums,1,0,None
1,artists,1,0,None
2,customers,11,0,None
3,employees,13,0,None
4,genres,1,0,None
5,invoice_items,2,2,"Quantity, UnitPrice"
6,invoices,7,1,Total
7,media_types,1,0,None
8,playlist_track,0,0,None
9,playlists,1,0,None


##### How many Discrete and Continuous Columns

In [17]:
# ============================================================
# HOW MANY DISCRETE AND CONTINUOUS COLUMNS? (first guess, by type)
# ------------------------------------------------------------
# Only numerical FEATURE columns (not IDs) are sorted here.
#   is_whole_number   -> BIGINT, INTEGER...  -> guess: DISCRETE
#   is_decimal_number -> DOUBLE, DECIMAL...  -> guess: CONTINUOUS
# The type is only a first guess. Query 2 checks it against the data.
#
# TO REUSE: change 'chinook' in all THREE places.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus two flags: whole number or decimal?
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT') as is_whole_number,
               data_type in ('FLOAT', 'DOUBLE', 'DECIMAL')               as is_decimal_number
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer: count and list each kind (feature columns only)
    select a.table_name,
           count(*) filter (where i.column_name is null and a.is_whole_number)   as n_discrete_columns,
           string_agg(a.column_name, ', ')
                    filter (where i.column_name is null and a.is_whole_number)   as discrete_columns,
           count(*) filter (where i.column_name is null and a.is_decimal_number) as n_continuous_columns,
           string_agg(a.column_name, ', ')
                    filter (where i.column_name is null and a.is_decimal_number) as continuous_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_discrete_columns,discrete_columns,n_continuous_columns,continuous_columns
0,albums,0,None,0,None
1,artists,0,None,0,None
2,customers,0,None,0,None
3,employees,0,None,0,None
4,genres,0,None,0,None
5,invoice_items,1,Quantity,1,UnitPrice
6,invoices,0,None,1,Total
7,media_types,0,None,0,None
8,playlist_track,0,None,0,None
9,playlists,0,None,0,None


##### Reality Check

In [18]:
# ============================================================
# REALITY CHECK: HOW MANY DIFFERENT VALUES DOES EACH
#                NUMERICAL COLUMN ACTUALLY HAVE?
# ------------------------------------------------------------
#   count(*)                -> how many rows
#   count(distinct column)  -> how many DIFFERENT values
# Few different values       -> behaves DISCRETE
# Close to one per row       -> behaves CONTINUOUS
#
# TO REUSE: one line per numerical column. On each line,
#           change the table name (2 places) and column name (2 places).
# ============================================================

con.sql("""
    select 'tracks'        as table_name, 'Milliseconds' as column_name,
           count(*)        as n_rows,
           count(distinct Milliseconds) as n_distinct_values
    from tracks
    union all
    select 'tracks',        'Bytes',     count(*), count(distinct Bytes)     from tracks
    union all
    select 'tracks',        'UnitPrice', count(*), count(distinct UnitPrice) from tracks
    union all
    select 'invoice_items', 'UnitPrice', count(*), count(distinct UnitPrice) from invoice_items
    union all
    select 'invoice_items', 'Quantity',  count(*), count(distinct Quantity)  from invoice_items
    union all
    select 'invoices',      'Total',     count(*), count(distinct Total)     from invoices
""").df()

,table_name,column_name,n_rows,n_distinct_values
0,tracks,Milliseconds,3503,3080
1,tracks,Bytes,3503,3501
2,tracks,UnitPrice,3503,2
3,invoice_items,UnitPrice,2240,2
4,invoice_items,Quantity,2240,1
5,invoices,Total,412,23


**Discrete / continuous — what the data shows:**
- Milliseconds (3,080 values) and Bytes (3,501) behave **continuous**,
  even though they're stored as whole numbers.
- UnitPrice (both tables) has only 2 values, so it behaves **discrete**.
- Quantity is always 1, a constant.
- Total has 23 values: discrete-ish.

**Rows with nulls in each:** 0 for all six numerical columns.

#### Categorical Columns

In [19]:
# ============================================================
# HOW MANY CATEGORICAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Categorical column = a FEATURE column (not an ID) stored as text.
#
# Same steps as the numerical query, with two more flags:
#   is_number   -> true if the type is a number type
#   is_category -> true if the type is text (labels)
#   is_date     -> true if the type is a date/time
# Every feature column should land in exactly ONE of these.
# CHECK: numerical + categorical + date = feature columns.
#
# TO REUSE: change 'chinook' in all THREE places.
#           Run the data-type query first; add any new type
#           names to the right flag list.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus three true/false flags for its kind
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT',
                             'FLOAT', 'DOUBLE', 'DECIMAL')   as is_number,
               data_type in ('VARCHAR', 'TEXT', 'BOOLEAN')   as is_category,
               data_type in ('DATE', 'TIMESTAMP', 'TIME')    as is_date
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer
    --   i.column_name is null = not an ID = a feature column
    select a.table_name,
           count(*) filter (where i.column_name is null)                   as n_feature_columns,
           count(*) filter (where i.column_name is null and a.is_number)   as n_numerical_columns,
           count(*) filter (where i.column_name is null and a.is_category) as n_categorical_columns,
           count(*) filter (where i.column_name is null and a.is_date)     as n_date_columns,
           string_agg(a.column_name, ', ')                                           -- list the names so you can see them
                    filter (where i.column_name is null and a.is_category) as categorical_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_feature_columns,n_numerical_columns,n_categorical_columns,n_date_columns,categorical_columns
0,albums,1,0,1,0,Title
1,artists,1,0,1,0,Name
2,customers,11,0,11,0,"Email, Company, State, Fax, LastName, City, Po..."
3,employees,13,0,11,2,"Phone, FirstName, Address, Country, Email, Tit..."
4,genres,1,0,1,0,Name
5,invoice_items,2,2,0,0,None
6,invoices,7,1,5,1,"BillingCity, BillingAddress, BillingCountry, B..."
7,media_types,1,0,1,0,Name
8,playlist_track,0,0,0,0,None
9,playlists,1,0,1,0,Name


#### Nominal / ordinal

In [20]:
# ============================================================
# LOOK AT THE VALUES: do these labels have a natural order?
# ------------------------------------------------------------
# Lists each different value and how many rows have it.
# Read the list and ask: "Is one of these higher/lower than another?"
#   No  -> NOMINAL
#   Yes -> ORDINAL
#
# TO REUSE: change the column name (3 places) and the table name.
# ============================================================

con.sql("""
    select Title,                          -- the column we're judging
           count(*) as n_employees         -- how many rows have each value
    from employees
    group by Title                         -- one row per different title
    order by n_employees desc              -- most common first
""").df()

,Title,n_employees
0,Sales Support Agent,3
1,IT Staff,2
2,General Manager,1
3,Sales Manager,1
4,IT Manager,1


**Rows with nulls in each:** [list the columns from the filtered
nulls output, e.g. tracks.Composer 978, customers.Company ..., ...]

In [21]:
# ============================================================
# HOW MANY NOMINAL AND ORDINAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Categorical column = feature column (not an ID) stored as text.
# Nominal vs ordinal can't be detected from the data type, so
# YOU decide, and list the ordinal ones in Step 5 by hand.
#   on the ordinal list  -> ORDINAL
#   not on the list      -> NOMINAL
#
# TO REUSE: change 'chinook' in all THREE places.
#           Edit the Step 5 list to match YOUR judgment.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus a flag: is it text (a category)?
    all_columns as (
        select table_name,
               column_name,
               data_type in ('VARCHAR', 'TEXT', 'BOOLEAN') as is_category
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    ),

    -- Step 5: YOUR judgment, a hand-made list of ordinal columns
    --   "values" builds a small table from rows you type.
    --   To add more: ('table', 'column'), ('table', 'column')
    ordinal_columns (table_name, column_name) as (
        values ('employees', 'Title')
    )

    -- Final answer
    --   i.column_name is null     = not an ID        (a feature)
    --   a.is_category             = stored as text   (categorical)
    --   o.column_name is not null = on YOUR ordinal list
    select a.table_name,
           count(*) filter (where i.column_name is null and a.is_category)                               as n_categorical_columns,
           count(*) filter (where i.column_name is null and a.is_category and o.column_name is null)     as n_nominal_columns,
           count(*) filter (where i.column_name is null and a.is_category and o.column_name is not null) as n_ordinal_columns,
           string_agg(a.column_name, ', ')
                    filter (where i.column_name is null and a.is_category and o.column_name is not null) as ordinal_columns
    from all_columns as a
    left join identifier_columns as i                 -- match against the ID list
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    left join ordinal_columns as o                    -- match against YOUR ordinal list
           on  a.table_name  = o.table_name
           and a.column_name = o.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_categorical_columns,n_nominal_columns,n_ordinal_columns,ordinal_columns
0,albums,1,1,0,None
1,artists,1,1,0,None
2,customers,11,11,0,None
3,employees,11,10,1,Title
4,genres,1,1,0,None
5,invoice_items,0,0,0,None
6,invoices,5,5,0,None
7,media_types,1,1,0,None
8,playlist_track,0,0,0,None
9,playlists,1,1,0,None


    **Ordinal:** 1 column, employees.Title. Job titles have a rank
    (managers above staff), recorded in ReportsTo.
    **Nominal:** every other text column.

    **Rows with nulls in each:** [list the columns from the filtered
    nulls output, e.g. tracks.Composer 978, customers.Company ..., ...]

    **Categories in each:** Title has 5. Columns like Name, Email, and
    Phone have almost one value per row, so they're unique labels,
    not true categories.

#### Nested Columns

**0 in every table**: SQLite has no nested data types

In [22]:
# ============================================================
# HOW MANY NESTED COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Nested column = one cell holds several values.
# In DuckDB, nested types look like:
#   INTEGER[]  or  VARCHAR[]   -> a LIST    (ends in [])
#   STRUCT(...)                -> a record  (several named fields)
#   MAP(...)                   -> key/value pairs
#   JSON                       -> JSON text
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           count(*) filter (where data_type like '%[]'          -- lists end in []
                               or data_type like 'STRUCT%'      -- records start with STRUCT
                               or data_type like 'MAP%'         -- maps start with MAP
                               or data_type = 'JSON')           -- JSON
               as n_nested_columns
    from information_schema.columns
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    group by table_name
    order by table_name
""").df()

,table_name,n_nested_columns
0,albums,0
1,artists,0
2,customers,0
3,employees,0
4,genres,0
5,invoice_items,0
6,invoices,0
7,media_types,0
8,playlist_track,0
9,playlists,0


## Breaking down the rows

### Rows with nulls

In [23]:
# ============================================================
# HOW MANY ROWS HAVE AT LEAST ONE NULL? (one table)
# ------------------------------------------------------------
# where columns(*) is not null -> keeps only rows where EVERY
#                                 column has a value (complete rows)
# rows with nulls = all rows - complete rows
#
# TO REUSE: change the table name (3 places).
# ============================================================

con.sql("""
    select count(*)                                                    as n_rows,
           (select count(*) from tracks where columns(*) is not null) as n_complete_rows,    -- every cell filled
           count(*) - (select count(*) from tracks where columns(*) is not null) as n_rows_with_nulls  -- at least one blank
    from tracks
""").df()

,n_rows,n_complete_rows,n_rows_with_nulls
0,3503,2525,978


In [24]:
# ============================================================
# ROWS WITH NULLS AND DUPLICATE ROWS, EVERY TABLE
# ------------------------------------------------------------
# Duplicates here compare the whole row (including the ID),
# so expect 0 everywhere. See "Duplicate Rows" below for the "ignore the ID" check.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

# 1. Ask SQL for the table names
tables = con.sql("""
    select table_name
    from information_schema.tables
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    order by table_name
""").df()["table_name"]

# 2. The pattern for ONE table. {t} is the blank for the table name.
pattern = """
    select '{t}' as table_name,
            count(*)                                                           as n_rows,
            count(*) - (select count(*) from {t} where columns(*) is not null) as n_rows_with_nulls,
            count(*) - (select count(*) from (select distinct * from {t}))      as n_duplicate_rows
    from {t}
"""

# 3. Fill in every table, stack with "union all", run once
query = "union all".join(pattern.format(t=t) for t in tables) + "order by table_name"
con.sql(query).df()

,table_name,n_rows,n_rows_with_nulls,n_duplicate_rows
0,albums,347,0,0
1,artists,275,0,0
2,customers,59,50,0
3,employees,8,1,0
4,genres,25,0,0
5,invoice_items,2240,0,0
6,invoices,412,209,0
7,media_types,5,0,0
8,playlist_track,8715,0,0
9,playlists,18,0,0


### Duplicate Rows

In [25]:
# ============================================================
# HOW MANY DUPLICATE ROWS? (one table, two ways)
# ------------------------------------------------------------
# distinct *                     -> one copy of each different row
# duplicates = all rows - distinct rows
#
# WAY A: compare ALL columns, including the ID
# WAY B: compare all columns EXCEPT the ID (more useful, see below)
#
# TO REUSE: change the table name (2 places in each)
#           and the ID column in WAY B.
# ============================================================

# WAY A: whole row, including the ID
con.sql("""
    select count(*)                                                     as n_rows,
           count(*) - (select count(*) from (select distinct * from tracks)) as n_duplicate_rows
    from tracks
""").df()

,n_rows,n_duplicate_rows
0,3503,0


In [26]:
# WAY B: ignore the ID column ("exclude" = every column except this one)
con.sql("""
    select count(*)                                                                       as n_rows,
           count(*) - (select count(*) from (select distinct * exclude (TrackId) from tracks)) as n_duplicate_rows
    from tracks
""").df()

,n_rows,n_duplicate_rows
0,3503,0


### Missing rows

In [27]:
# ============================================================
# MISSING ROWS, READING A: gaps in the ID sequence
# ------------------------------------------------------------
# expected rows = last ID - first ID + 1
# missing rows  = expected rows - actual rows
# (Assumes IDs are numbered 1, 2, 3... with no planned gaps.)
#
# TO REUSE: change the ID column (4 places) and the table name.
# ============================================================

con.sql("""
    select min(TrackId)                           as first_id,
           max(TrackId)                           as last_id,
           count(*)                               as n_rows,
           max(TrackId) - min(TrackId) + 1        as n_expected_rows,
           max(TrackId) - min(TrackId) + 1 - count(*) as n_missing_rows
    from tracks
""").df()

,first_id,last_id,n_rows,n_expected_rows,n_missing_rows
0,1,3503,3503,3503,0


In [28]:
# ============================================================
# MISSING ROWS, READING B: foreign key points to nothing
# ------------------------------------------------------------
# left join keeps every track. If no album matches,
# the album side is blank (null): that album row is MISSING.
#
# TO REUSE: change both tables and the key column.
# ============================================================

con.sql("""
    select count(*) as n_tracks_with_missing_album
    from tracks as t
    left join albums as a
           on t.AlbumId = a.AlbumId
    where a.AlbumId is null              -- no matching album found
      and t.AlbumId is not null          -- (ignore tracks that simply have no album listed)
""").df()

,n_tracks_with_missing_album
0,0


### Rows: summary
- **Rows with nulls:** customers 50, employees 1, invoices 209, tracks 978 (all from Composer). Every other table: 0.
- **Duplicate rows:** 0 in every table; 0 duplicate tracks even ignoring TrackId.
- **Missing rows:** 0 gaps in TrackId (1–3503); 0 tracks pointing to a missing album.